In [ ]:
!find . -name "__pycache__" -exec rm -rf {} + 2>/dev/null; echo "caches cleared"
!cat gaussianfractallod/data.py | grep -A2 "flip"

In [ ]:
# Step 1: Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies and clone repo
!pip install -q gsplat torchmetrics lpips tensorboard pyyaml

import os
if not os.path.exists('/content/GaussianFractalLOD'):
    !git clone https://github.com/dedoubleyou1/GaussianFractalLOD.git

%cd /content/GaussianFractalLOD

# Pull LFS data (NeRF Synthetic scenes)
!apt-get install -y git-lfs -qq
!git lfs install
!git lfs pull

!pip install -e . -q
print('Setup complete!')

In [ ]:
!git pull && pip install -e .

In [ ]:
# NeRF Synthetic data is included in the repo (via Git LFS)
# All 8 scenes: chair, drums, ficus, hotdog, lego, materials, mic, ship
import os
REPO_DIR = os.getcwd()  # Should be /content/GaussianFractalLOD
DATA_DIR = os.path.join(REPO_DIR, 'nerf_synthetic')
assert os.path.exists(DATA_DIR), f'Data not found at {DATA_DIR}'
!ls {DATA_DIR}/

In [ ]:
from gaussianfractallod.config import Config

SCENE = "lego"  # Change for different scenes

cfg = Config(
    data_dir=f"{DATA_DIR}/{SCENE}",
    num_roots=1,
    sh_degree=3,
    max_binary_depth=18,
    root_iterations=5000,
    split_iterations_per_level=3000,
    split_convergence_window=500,
    checkpoint_dir=f"/content/drive/MyDrive/GaussianFractalLOD/checkpoints/{SCENE}",
)
print(f"Training {SCENE} with {cfg.num_roots} root, SH{cfg.sh_degree}, depth {cfg.max_binary_depth}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from gaussianfractallod.train import train
from pathlib import Path

# To resume from a checkpoint, set this path:
RESUME_FROM = None  # e.g., f"{cfg.checkpoint_dir}/phase2_level_3.pt"

roots, tree = train(cfg, resume_from=RESUME_FROM)
print(f"Training complete! {tree.num_splits} splits across {tree.depth} levels")

In [ ]:
import torch
from gaussianfractallod.data import NerfSyntheticDataset
from gaussianfractallod.eval import evaluate

device = torch.device("cuda")
test_dataset = NerfSyntheticDataset(cfg.data_dir, split="test")
background = torch.tensor(cfg.background_color, device=device)

results = {}
for depth in range(tree.depth + 1):
    r = evaluate(roots.to(device), tree.to(device), test_dataset, depth, device, background)
    results[depth] = r
    print(f"Depth {depth}: PSNR={r['psnr']:.2f}, {r['num_gaussians']} Gaussians")

In [ ]:
import torch
import matplotlib.pyplot as plt
from gaussianfractallod.reconstruct import reconstruct
from gaussianfractallod.render import render_gaussians
from gaussianfractallod.data import NerfSyntheticDataset

device = torch.device("cuda")
test_dataset = NerfSyntheticDataset(cfg.data_dir, split="test")
background = torch.tensor(cfg.background_color, device=device)

# Pick a test view
gt_image, camera = test_dataset[0]
cam = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in camera.items()}

# Sample depths evenly across the tree, always include 0 and max
max_d = tree.depth
if max_d <= 8:
    depths = list(range(max_d + 1))
else:
    # Show ~8 evenly spaced depths plus 0 and max
    depths = sorted(set([0] + list(range(0, max_d + 1, max(1, max_d // 7))) + [max_d]))

ncols = len(depths) + 1  # +1 for ground truth
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4), dpi=150)

# Ground truth
axes[0].imshow(gt_image.numpy().clip(0, 1))
axes[0].set_title("Ground Truth", fontsize=10)
axes[0].axis("off")

for i, depth in enumerate(depths):
    with torch.no_grad():
        gaussians = reconstruct(roots.to(device), tree.to(device), depth)
        rendered = render_gaussians(
            gaussians, cam["viewmat"], cam["K"],
            cam["width"], cam["height"], background,
        )
    axes[i + 1].imshow(rendered.cpu().numpy().clip(0, 1))
    axes[i + 1].set_title(f"D{depth}: {gaussians.num_gaussians:,} G", fontsize=10)
    axes[i + 1].axis("off")

plt.suptitle(f"LoD Progression — {SCENE} (SH{cfg.sh_degree}, {cfg.num_roots} root)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{cfg.checkpoint_dir}/lod_progression.png", dpi=150, bbox_inches="tight")
plt.show()

# Also plot PSNR vs depth curve
if results:
    fig2, ax2 = plt.subplots(1, 1, figsize=(8, 5), dpi=150)
    ds = sorted(results.keys())
    psnrs = [results[d]["psnr"] for d in ds]
    counts = [results[d]["num_gaussians"] for d in ds]

    ax2.plot(ds, psnrs, "o-", color="steelblue", linewidth=2)
    ax2.set_xlabel("Binary Depth")
    ax2.set_ylabel("PSNR (dB)")
    ax2.set_title(f"PSNR vs Depth — {SCENE}")
    ax2.grid(True, alpha=0.3)

    # Add Gaussian count labels
    for d, p, c in zip(ds, psnrs, counts):
        if d % 3 == 0 or d == max(ds):
            ax2.annotate(f"{c:,}", (d, p), textcoords="offset points",
                        xytext=(0, 10), ha="center", fontsize=7, color="gray")

    plt.tight_layout()
    plt.savefig(f"{cfg.checkpoint_dir}/psnr_vs_depth.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Resume from depth 6 checkpoint
import glob
ckpts = sorted(glob.glob(f"{cfg.checkpoint_dir}/phase2_level_*.pt"))
RESUME_FROM = ckpts[-1] if ckpts else None
print(f"Resuming from: {RESUME_FROM}")

roots, tree = train(cfg, resume_from=RESUME_FROM)

In [ ]:
from gaussianfractallod.reconstruct import reconstruct
from gaussianfractallod.export_ply import export_ply

# Export at different LoD levels
for depth in [0, 3, 6, 9, 12, 15, 18]:
    if depth > tree.depth:
        break
    with torch.no_grad():
        gaussians = reconstruct(roots.to("cuda"), tree.to("cuda"), depth)
    export_ply(gaussians, f"/content/drive/MyDrive/GaussianFractalLOD/exports/lego_depth_{depth}.ply", sh_degree=cfg.sh_degree)

print("\nDrag any .ply file into https://antimatter15.com/splat/ to view in 3D")